# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import zip
from pathlib import Path
import torch
from hydra.utils import instantiate

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils.notebook_setup import init_nlp_notebook # noqa: E402
cfg = init_nlp_notebook()
device = "cuda" if torch.cuda.is_available() else "cpu"

# Generation core Init

In [ ]:
from src.core.models.generation import HFTextGenerator

# Обязательно левый паддинг для батчевой генерации!
cfg.model.tokenizer.padding_side = "left"
tokenizer = instantiate(cfg.model.tokenizer).build()

# Загружаем модель
model_builder = instantiate(cfg.model.builder, tokenizer=tokenizer)
model = model_builder.build()

# Инициализируем генератор (с клинером внутри)
generator = HFTextGenerator(
    model=model,
    tokenizer=tokenizer,
    generation_kwargs=cfg.model.generation_kwargs,
    cleaner_cfg=cfg.model.get("cleaner")
) #

# Hyperparameters

In [ ]:
from src.core.models.promts import PromptManager

test_query = "Объясни концепцию 'Attention is all you need' в трех предложениях."
prompt = PromptManager.build_simple_prompt(test_query) #

# Экспериментируем с температурой на одном и том же промпте
experiments = [
    {"temperature": 0.1, "do_sample": True, "top_p": 0.9}, # Строгий и детерминированный
    {"temperature": 0.7, "do_sample": True, "top_p": 0.9}, # Креативный (стандарт)
    {"temperature": 1.5, "do_sample": True, "top_p": 0.95} # Максимальная галлюцинация
]

print(f"PROMPT:\n{prompt}\n")

for i, kwargs in enumerate(experiments):
    # Твой класс аккуратно смержит эти kwargs с дефолтными
    response = generator.generate(texts=[prompt], max_new_tokens=150, **kwargs)[0] #[cite: 13]
    
    print(f"--- Experiment {i+1} (Temp: {kwargs['temperature']}) ---")
    print(response)
    print()

# Few-Shot Prompting

In [ ]:
# Набрасываем Few-Shot примеры прямо здесь
few_shot_template = """Ты — экстрактор сущностей. Извлекай компании и возвращай ТОЛЬКО валидный JSON.

Пример 1:
Текст: Вчера Apple анонсировала новый iPhone, а акции Microsoft выросли.
Ответ: {{"companies": ["Apple", "Microsoft"]}}

Пример 2:
Текст: Илон Маск купил Twitter за 44 миллиарда.
Ответ: {{"companies": ["Twitter"]}}

Текст: {query}
Ответ:"""

test_texts = [
    "Google и Amazon инвестируют в стартапы в сфере AI.",
    "Вчера сходил в магазин за хлебом."
]

prompts = [few_shot_template.format(query=t) for t in test_texts]

# Генерируем с нулевой температурой, чтобы формат не "поплыл"
responses = generator.generate(prompts, max_new_tokens=50, temperature=0.1, do_sample=True) #[cite: 13]

for t, r in zip(test_texts, responses):
    print(f"TEXT: {t}")
    print(f"EXTRACTED: {r}")
    print("-" * 30)

# Full RAG Pipeline Test

In [ ]:
from src.core.rag.retriever import RAGRetriever

# 1. Поднимаем ретривер
try:
    retriever = instantiate(cfg.rag.retriever)
except Exception as e:
    print(f"Векторная база не найдена! Убедись, что запустил 02_retrieval_experiments.ipynb\n{e}")
    retriever = None

if retriever:
    rag_query = "Как очистить текст от HTML тегов?"
    
    # 2. Достаем контекст
    context = retriever.retrieve_context(rag_query) #[cite: 8]
    
    # 3. Собираем промпт через твой менеджер
    rag_prompt = PromptManager.build_rag_prompt(query=rag_query, context=context) #[cite: 15]
    
    # 4. Генерируем ответ
    rag_response = generator.generate([rag_prompt], max_new_tokens=200, temperature=0.3)[0] #[cite: 13]
    
    print(f"QUERY: {rag_query}\n")
    print(f"CONTEXT EXTRACTED:\n{context[:300]}...\n")
    print(f"MODEL RESPONSE:\n{rag_response}")

# Edge Cases & Cleaner Audit

In [ ]:
from src.core.models.parsers import ResponseCleaner

dirty_query = "Расскажи сказку."
prompt = PromptManager.build_simple_prompt(dirty_query) #[cite: 15]

# Намеренно обрываем генерацию на полуслове (max_new_tokens=15)
raw_responses = generator.generate(
    [prompt], 
    max_new_tokens=15, 
    temperature=0.7, 
    do_sample=True
) #[cite: 13]

# В HFTextGenerator уже встроен ResponseCleaner
# По умолчанию trim_incomplete_sentence=True, поэтому он должен срезать оборванное предложение. #[cite: 14]
print(f"CLEANED TRUNCATED RESPONSE:\n{raw_responses[0]}")